In [0]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))

if project_root not in sys.path:
    sys.path.append(project_root)

from pyspark.sql.functions import col, when, timestamp_diff
from pyspark.sql.types import DoubleType, TimestampType
from modules.data.bulk_read import get_merged_df
from modules.utils.date import get_months_start_n_months_ago

In [0]:
# Append the latest bike trips records, cleaned to silver layer
# 1. Get a dataframe with records from all cities, from bronze layer
# 2. Select and cast columns of filtered rows with latests ride trips and with start and end stations defined
# 3. Append data to silver layer cleansed table
target_month = int(dbutils.widgets.get("months_ago"))
target_month_start = get_months_start_n_months_ago(months_ago=target_month)
next_month_start = get_months_start_n_months_ago(months_ago=target_month - 1)

In [0]:
df = get_merged_df(spark=spark, catalog="bikes", schema="01_bronze", substring="_trips_raw")

In [0]:
df = df.filter(f"started_at >= '{target_month_start}' AND started_at < '{next_month_start}'").\
        filter((col("start_station_id").isNotNull()) & (col("end_station_id").isNotNull())).\
        select(
        col("city"),
        col("ride_id"),
        col("rideable_type"),
        col("started_at").cast(TimestampType()),
        col("ended_at").cast(TimestampType()),
        col("start_station_name"),
        col("start_station_id"),
        col("end_station_name"),
        col("end_station_id"),
        col("start_lat").cast(DoubleType()),
        col("start_lng").cast(DoubleType()),
        col("end_lat").cast(DoubleType()),
        col("end_lng").cast(DoubleType()),
        col("member_casual"),
        col("processed_timestamp")
        )

In [0]:
print(df.count())

In [0]:
df.write.mode("append").saveAsTable("bikes.02_silver.trips_cleansed")